In [29]:
import os, json
%reload_ext dotenv
%dotenv
AWS_BEARER_TOKEN_BEDROCK = os.getenv('AWS_BEARER_TOKEN_BEDROCK', None)
INFERENCE_PROFILE_ARN=os.getenv('AWS_BEDROCK_INFERENCE_PROFILE_ARN', None)

In [35]:
import boto3
import json
import os
from botocore.config import Config

def setup_bedrock_client():
    """
    Set up the Bedrock client using bearer token authentication
    """
    # Get credentials from environment variables
    bearer_token = os.getenv('AWS_BEARER_TOKEN_BEDROCK')
    inference_profile_arn = os.getenv('AWS_BEDROCK_INFERENCE_PROFILE_ARN')
    
    if not bearer_token or not inference_profile_arn:
        raise ValueError("Please set AWS_BEARER_TOKEN_BEDROCK and AWS_BEDROCK_INFERENCE_PROFILE_ARN environment variables")
    
    # Configure the client
    config = Config(
        region_name='us-east-1',  # or your preferred region
        retries={'max_attempts': 3}
    )
    
    # Create Bedrock Runtime client
    bedrock_client = boto3.client(
        'bedrock-runtime',
        config=config,
        aws_access_key_id='dummy',  # Required but not used with bearer token
        aws_secret_access_key='dummy',  # Required but not used with bearer token
    )
    
    return bedrock_client, inference_profile_arn

def generate_summary(text_to_summarize, max_length=200, prompt = None):
    """
    Generate a summary using Claude Sonnet via AWS Bedrock
    """
    client, inference_profile_arn = setup_bedrock_client()
    
    # Prepare the prompt
    if prompt:
        print("using provided proompt***")
    else:
        prompt = f"""
        Please provide a concise summary of the following text in approximately {max_length} words or less:

        {text_to_summarize}

        Summary:"""
    
    # Prepare the request body for Claude
    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_length * 2,  # Rough estimate for token count
        "temperature": 0.1,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ]
    }
    
    try:
        # Make the API call
        response = client.invoke_model(
            modelId=inference_profile_arn,
            body=json.dumps(request_body),
            contentType='application/json',
            accept='application/json'
        )
        
        # Parse the response
        response_body = json.loads(response['body'].read())
        summary = response_body['content'][0]['text']
        
        return summary.strip()
        
    except Exception as e:
        print(f"Error generating summary: {str(e)}")
        return None

# Example usage
def main():
    # Sample text to summarize
    sample_text = """
    Artificial intelligence (AI) has become increasingly prevalent in modern society, 
    transforming industries from healthcare to finance. Machine learning algorithms 
    can now process vast amounts of data to identify patterns and make predictions 
    that were previously impossible for humans to detect. However, this rapid 
    advancement also raises important questions about privacy, job displacement, 
    and the ethical implications of automated decision-making. As we continue to 
    integrate AI into our daily lives, it's crucial to establish frameworks that 
    ensure these technologies are developed and deployed responsibly, with proper 
    consideration for their societal impact.
    """
    
    # Generate summary
    summary = generate_summary(sample_text, max_length=100)
    
    if summary:
        print("Original text:")
        print(sample_text)
        print("\n" + "="*50)
        print("Generated Summary:")
        print(summary)
    else:
        print("Failed to generate summary")

if __name__ == "__main__":
    main()


Original text:

    Artificial intelligence (AI) has become increasingly prevalent in modern society, 
    transforming industries from healthcare to finance. Machine learning algorithms 
    can now process vast amounts of data to identify patterns and make predictions 
    that were previously impossible for humans to detect. However, this rapid 
    advancement also raises important questions about privacy, job displacement, 
    and the ethical implications of automated decision-making. As we continue to 
    integrate AI into our daily lives, it's crucial to establish frameworks that 
    ensure these technologies are developed and deployed responsibly, with proper 
    consideration for their societal impact.
    

Generated Summary:
AI has rapidly transformed industries like healthcare and finance through machine learning algorithms that can process large datasets and identify patterns beyond human capability. However, this technological advancement presents significant challeng

In [ ]:
# Sample content combining text and a simple table (as plain text).
REPORT = textwrap.dedent("""\
Quarterly Sales Performance Report (Q3 2025)

Narrative:
In the third quarter of 2025, the company experienced significant growth in its consumer electronics division,
driven by strong demand for wearable devices and smart home products. Despite supply chain challenges, overall
revenue increased, and customer satisfaction scores improved. The marketing campaign launched in July contributed
to a notable rise in new customer acquisitions. The company plans to expand its product line in the next quarter
to maintain momentum.

Sales Data by Product Category (Q3 2025)
| Product Category     | Units Sold | Revenue (£) | Customer Satisfaction (%) |
|----------------------|------------|-------------|----------------------------|
| Wearable Devices     | 12,400     | 3,100,000   | 92                         |
| Smart Home Products  | 8,700      | 2,050,000   | 89                         |
| Mobile Accessories   | 5,200      | 780,000     | 85                         |
| Audio Equipment      | 4,600      | 1,200,000   | 88                         |

Customer feedback indicates that the new range of wearable devices, particularly the fitness trackers, received
positive reviews for their accuracy and battery life. Smart home products saw a surge in popularity as more customers
invested in home automation.
""")

PROMPT = textwrap.dedent(f"""\
You are a highly efficient text summarizer.
- Read the narrative and the table.
- Produce a concise executive summary (4–8 bullet points).
- Include 1–2 key quantitative highlights (units sold, revenue, satisfaction).
- Mention notable trends or changes and any planned next steps.
- Be faithful to the data; do not infer facts not present.
Text to summarize:
{REPORT}
""")
summary = generate_summary(REPORT, max_length=100, prompt=PROMPT)
print(summary)

using provided proompt***
I don't see any narrative text or table provided in your message. You've shared the instructions for creating an executive summary, but the actual content to summarize (the "{text_to_summarize}" section) appears to be empty or wasn't included.

Could you please provide:
1. The narrative text you'd like me to summarize
2. The table with relevant data

Once you share this content, I'll create a concise executive summary following your specifications with 4-8 bullet points, key quantitative highlights, trends, and next steps as requested.
